# 04 - Ingestão Bronze: Micro-Lotes em Delta Lake com Metadados
**Squad 2 — Real Time for Business | Dupla 1**  
**Integrantes:** Lucas Sousa Santos Oliveira & Zaiden Emiliano Segundo Seleme  
**Tabelas de Escopo:** `ecommerce_produtos` e `ecommerce_categorias`  
**Branch:** `feat/squad2-lucas_zaiden`  

### Objetivo da Task (Sprint 2 - Task 1):
1. **Detecção Incremental via Loop & Controle de Estado:** Identificar novos arquivos `.parquet` depositados no bucket `raw/real-time-data/` verificando contra a tabela de controle Delta.
2. **Auditoria e Rastreabilidade:** Adicionar em cada registro as colunas obrigatórias:
   * `bronze_ingested_at`: timestamp do momento da ingestão via `current_timestamp()`.
   * `bronze_source_file`: caminho completo do arquivo de origem via `lit(arquivo_path)`.
3. **Zero Transformação (Regra de Ouro Bronze):** Não aplicar limpeza, filtros de regras de negócio ou alteração de tipos. O dado bruto é preservado 100% íntegro.
4. **Particionamento Temporal Obrigatório:** Salvar em formato **Delta Lake** em modo **`append`**, particionado por data de ingestão (`ano`, `mes`, `dia`, `hora`).
5. **Isolamento de Namespace (`/grupo1/`):** Gravar os dados no container `squad2` sob o prefixo seguro `/grupo1/` para evitar conflito com outras duplas.
6. **Controle de Idempotência e Metadados:** Registrar os arquivos e lotes processados na tabela Delta de controle `squad2.ingestion_control_log`.

## 1. Carregamento Seguro das Credenciais e Instalação de Dependências

In [0]:
# Garantir dependências para listagem segura via Service Principal
%pip install -q python-dotenv azure-storage-file-datalake azure-identity

import os
from dotenv import load_dotenv, find_dotenv
from azure.identity import ClientSecretCredential
from azure.storage.file.datalake import DataLakeServiceClient

# Carregamento automático do arquivo .env com override
dotenv_path = find_dotenv()
if not dotenv_path:
    candidatos = [
        os.path.join(os.getcwd(), ".env"),
        os.path.join(os.path.dirname(os.getcwd()), ".env"),
        os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), ".env")
    ]
    for c in candidatos:
        if os.path.exists(c):
            dotenv_path = c
            break

load_dotenv(dotenv_path, override=True)

storage_account = os.getenv("ADLS_STORAGE_ACCOUNT_NAME", "internshipdatalake")
client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")

print("Ambiente configurado com sucesso:")
print(f"  Storage Account: {storage_account}")
print(f"  Client ID disponível: {client_id is not None}")
print(f"  Tenant ID disponível: {tenant_id is not None}")
print(f"  Client Secret disponível: {client_secret is not None}")

## 2. Configurações de Conexão OAuth e Definição dos Caminhos ABFSS
Injetamos as credenciais via `adls_options` diretamente nos leitores e escritores do Spark, compatível 100% com Databricks Serverless.

In [0]:
# Configurações OAuth do Service Principal para injeção granular nas operações do cluster
adls_options = {
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net": client_id,
    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net": client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

# Definição dos caminhos com isolamento de namespace no prefixo /grupo1/
base_raw = f"abfss://raw@{storage_account}.dfs.core.windows.net/real-time-data"
base_squad = f"abfss://squad2@{storage_account}.dfs.core.windows.net/grupo1"

caminhos = {
    "bronze_produtos": f"{base_squad}/bronze/ecommerce_produtos",
    "bronze_categorias": f"{base_squad}/bronze/ecommerce_categorias",
    "metadata_control_log": f"{base_squad}/metadata/ingestion_control_log"
}

print("Caminhos configurados no Data Lake:")
for k, v in caminhos.items():
    print(f"  {k}: {v}")

## 3. Função de Detecção de Arquivos Novos (Controle de Estado)
Varre o bucket `raw/real-time-data` via API do ADLS Gen2 e consulta a tabela Delta de controle `squad2.ingestion_control_log` para retornar apenas arquivos que ainda não foram processados.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, LongType
from pyspark.sql.functions import current_timestamp, year, month, dayofmonth, hour, col, lit

def listar_arquivos_novos(nome_tabela):
    """
    Identifica arquivos .parquet novos no bucket raw que ainda não constam na tabela de controle.
    """
    # 1. Obter lista de arquivos já processados da tabela de controle Delta
    arquivos_ja_processados = set()
    try:
        df_control = spark.read.format("delta").options(**adls_options).load(caminhos["metadata_control_log"])
        linhas = df_control.filter(col("tabela_origem") == nome_tabela).select("arquivo_processado").distinct().collect()
        arquivos_ja_processados = {r["arquivo_processado"] for r in linhas}
    except Exception:
        # Primeira execução (tabela de controle ainda não existe)
        pass

    # 2. Conectar via Azure SDK REST para listar os arquivos do bucket raw
    credencial = ClientSecretCredential(tenant_id=tenant_id, client_id=client_id, client_secret=client_secret)
    service_client = DataLakeServiceClient(account_url=f"https://{storage_account}.dfs.core.windows.net", credential=credencial)
    file_system = service_client.get_file_system_client("raw")

    sufixo_alvo = f"{nome_tabela}.parquet"
    arquivos_novos = []

    print(f"Varrendo bucket 'raw/real-time-data' em busca de arquivos '{sufixo_alvo}'...")
    caminhos_encontrados = file_system.get_paths(path="real-time-data")

    for p in caminhos_encontrados:
        if not p.is_directory and p.name.endswith(sufixo_alvo):
            caminho_abfss = f"abfss://raw@{storage_account}.dfs.core.windows.net/{p.name}"
            if caminho_abfss not in arquivos_ja_processados:
                arquivos_novos.append(caminho_abfss)

    print(f"-> Arquivos totais já processados: {len(arquivos_ja_processados)}")
    print(f"-> Novos arquivos detectados para ingestão: {len(arquivos_novos)}")
    return sorted(arquivos_novos)

print("Função listar_arquivos_novos compilada com sucesso.")

## 4. Ingestão Bronze: `ecommerce_produtos`
Para cada arquivo novo detectado pelo loop:
1. Lê o `.parquet` com PySpark distribuído.
2. Adiciona colunas de auditoria: `bronze_ingested_at` e `bronze_source_file`.
3. Deriva as colunas de partição: `ano`, `mes`, `dia`, `hora`.
4. Salva como **Delta Lake** na Bronze em modo **`append`** particionado por data de ingestão.
5. Registra o arquivo na tabela Delta de controle.

In [0]:
arquivos_produtos = listar_arquivos_novos("ecommerce_produtos")

if not arquivos_produtos:
    print("Nenhum arquivo novo para processar em 'ecommerce_produtos'. Camada Bronze está atualizada!")
else:
    print(f"\nIniciando ingestão de {len(arquivos_produtos)} micro-lotes para 'ecommerce_produtos'...")
    
    schema_log = StructType([
        StructField("tabela_origem", StringType(), False),
        StructField("arquivo_processado", StringType(), False),
        StructField("total_linhas", LongType(), False),
        StructField("status", StringType(), False)
    ])

    for idx, arq in enumerate(arquivos_produtos, start=1):
        print(f"\n[{idx}/{len(arquivos_produtos)}] Lendo: {arq}")
        
        # 1. Leitura distribuída do arquivo Parquet bruto
        df_raw = spark.read.options(**adls_options).parquet(arq)
        total_linhas = df_raw.count()
        
        # 2. Adicionar colunas de auditoria obrigatórias e partições temporais (ZERO transformação no dado bruto)
        df_bronze = (df_raw
            .withColumn("bronze_ingested_at", current_timestamp())
            .withColumn("bronze_source_file", lit(arq))
            .withColumn("ano", year(col("bronze_ingested_at")))
            .withColumn("mes", month(col("bronze_ingested_at")))
            .withColumn("dia", dayofmonth(col("bronze_ingested_at")))
            .withColumn("hora", hour(col("bronze_ingested_at"))))
        
        # 3. Salvar em Delta Bronze particionado em modo append
        (df_bronze.write
            .format("delta")
            .options(**adls_options)
            .mode("append")
            .partitionBy("ano", "mes", "dia", "hora")
            .save(caminhos["bronze_produtos"]))
        
        # 4. Registrar na tabela de controle de auditoria
        df_log = (spark.createDataFrame([( "ecommerce_produtos", arq, int(total_linhas), "SUCCESS" )], schema=schema_log)
                  .withColumn("timestamp_processamento", current_timestamp()))
        
        (df_log.write
            .format("delta")
            .options(**adls_options)
            .mode("append")
            .save(caminhos["metadata_control_log"]))
        
        print(f"    -> Ingeridas {total_linhas} linhas com sucesso na Bronze e auditadas no controle.")
    
    print("\nIngestão Bronze de 'ecommerce_produtos' concluída com sucesso!")

## 5. Ingestão Bronze: `ecommerce_categorias`
Executa o loop incremental para processar os arquivos de categorias com auditoria e particionamento temporal.

In [0]:
arquivos_categorias = listar_arquivos_novos("ecommerce_categorias")

if not arquivos_categorias:
    print("Nenhum arquivo novo para processar em 'ecommerce_categorias'. Camada Bronze está atualizada!")
else:
    print(f"\nIniciando ingestão de {len(arquivos_categorias)} micro-lotes para 'ecommerce_categorias'...")
    
    schema_log = StructType([
        StructField("tabela_origem", StringType(), False),
        StructField("arquivo_processado", StringType(), False),
        StructField("total_linhas", LongType(), False),
        StructField("status", StringType(), False)
    ])

    for idx, arq in enumerate(arquivos_categorias, start=1):
        print(f"\n[{idx}/{len(arquivos_categorias)}] Lendo: {arq}")
        
        # 1. Leitura distribuída do arquivo Parquet bruto
        df_raw = spark.read.options(**adls_options).parquet(arq)
        total_linhas = df_raw.count()
        
        # 2. Adicionar colunas de auditoria obrigatórias e partições temporais (ZERO transformação no dado bruto)
        df_bronze = (df_raw
            .withColumn("bronze_ingested_at", current_timestamp())
            .withColumn("bronze_source_file", lit(arq))
            .withColumn("ano", year(col("bronze_ingested_at")))
            .withColumn("mes", month(col("bronze_ingested_at")))
            .withColumn("dia", dayofmonth(col("bronze_ingested_at")))
            .withColumn("hora", hour(col("bronze_ingested_at"))))
        
        # 3. Salvar em Delta Bronze particionado em modo append
        (df_bronze.write
            .format("delta")
            .options(**adls_options)
            .mode("append")
            .partitionBy("ano", "mes", "dia", "hora")
            .save(caminhos["bronze_categorias"]))
        
        # 4. Registrar na tabela de controle de auditoria
        df_log = (spark.createDataFrame([( "ecommerce_categorias", arq, int(total_linhas), "SUCCESS" )], schema=schema_log)
                  .withColumn("timestamp_processamento", current_timestamp()))
        
        (df_log.write
            .format("delta")
            .options(**adls_options)
            .mode("append")
            .save(caminhos["metadata_control_log"]))
        
        print(f"    -> Ingeridas {total_linhas} linhas com sucesso na Bronze e auditadas no controle.")
    
    print("\nIngestão Bronze de 'ecommerce_categorias' concluída com sucesso!")

## 6. Criação e Mapeamento das Tabelas Externas no Databricks Metastore
Mapeia as tabelas Bronze sob o schema `squad2` apontando para o namespace seguro `/grupo1/` no Data Lake.

In [0]:
try:
    spark.sql("CREATE SCHEMA IF NOT EXISTS squad2")

    # Registrar tabela externa de produtos
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS squad2.bronze_ecommerce_produtos
    USING DELTA
    LOCATION '{caminhos["bronze_produtos"]}'
    """)

    # Registrar tabela externa de categorias
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS squad2.bronze_ecommerce_categorias
    USING DELTA
    LOCATION '{caminhos["bronze_categorias"]}'
    """)

    # Registrar tabela externa de controle de ingestão
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS squad2.ingestion_control_log
    USING DELTA
    LOCATION '{caminhos["metadata_control_log"]}'
    """)
    print("Tabelas externas registradas com sucesso no schema squad2!")
except Exception as e:
    print(f"Nota sobre Metastore Externo: {e}")
    print("Os dados Delta no ADLS e views temporárias estão prontos para consulta local.")

## 7. Auditoria Pós-Carga e Validação das Partições Físicas

In [0]:
print("=== Auditoria da Camada Bronze ===")

# 1. Leitura direta via Delta no Data Lake com autenticação granular adls_options
df_check_prod = spark.read.format("delta").options(**adls_options).load(caminhos["bronze_produtos"])
df_check_cat = spark.read.format("delta").options(**adls_options).load(caminhos["bronze_categorias"])
df_check_log = spark.read.format("delta").options(**adls_options).load(caminhos["metadata_control_log"])

# Criar views temporárias para consultas SQL diretas
df_check_prod.createOrReplaceTempView("bronze_ecommerce_produtos")
df_check_cat.createOrReplaceTempView("bronze_ecommerce_categorias")
df_check_log.createOrReplaceTempView("ingestion_control_log")

count_prod = df_check_prod.count()
count_cat = df_check_cat.count()
count_log = df_check_log.count()

print(f"Total de registros em bronze_ecommerce_produtos:   {count_prod}")
print(f"Total de registros em bronze_ecommerce_categorias: {count_cat}")
print(f"Total de registros na tabela de controle de logs:   {count_log}")

# 2. Verificação das partições temporais criadas
print("\n--- Partições Temporais Detectadas em Produtos ---")
display(df_check_prod.groupBy("ano", "mes", "dia", "hora").count().orderBy("ano", "mes", "dia", "hora"))

# 3. Exibir amostra dos metadados de auditoria
print("\n--- Amostra de Registros da Camada Bronze (Produtos) ---")
display(df_check_prod.select("sku", "nome_produto", "preco_lista", "bronze_ingested_at", "bronze_source_file").limit(5))

# 4. Exibir o histórico da tabela de controle de arquivos
print("\n--- Histórico de Ingestão (ingestion_control_log) ---")
display(df_check_log.orderBy(col("timestamp_processamento").desc()))